# 02 · Choose a method that fits your GPU

**What you will do**
1. Ask Training Hub's **memory estimator** how much GPU memory each method needs, before booking a GPU.
2. Meet **OSFT**, Red Hat's full fine-tuning method that protects what the model already knows.
3. Pick the method by the mistake the model makes.

**Time:** 5 minutes. **GPU:** none.

| | Dev environment | On OpenShift AI |
|---|---|---|
| Will it fit? | Run until out of memory | Memory estimator, per method, model and GPU |
| Methods | A different library and config each | One Training Hub call each: `sft`, `osft`, `lora_sft`, `lora_grpo`, `embedding_sft` |
| Keeping general skills | No protection | `osft` with one setting, `unfreeze_rank_ratio` |

In [ ]:
# Settings used by every notebook. Change them here if your project differs.
import os, json, time
from pathlib import Path

NAMESPACE  = open("/var/run/secrets/kubernetes.io/serviceaccount/namespace").read().strip()   # this project
WS         = Path("/opt/app-root/src/workspace")   # the project volume, as mounted in this workbench
PVC        = "workspace"                           # its name; training and serving pods mount it at /workspace
REPO       = Path.cwd().parent                     # this repository (data/, sdg_flows/)
BASE_MODEL = "ibm-granite/granite-4.1-3b"
EXPERIMENT = "sentinel7"                           # MLflow experiment
NODE_SELECTOR = {}   # only with ReadWriteOnce storage: keep every pod that mounts the volume on one node
TRAINING_IMAGE = ("registry.redhat.io/rhoai/odh-th-torch-cuda-py312-rhel9@sha256:"   # OpenShift AI 3.5 Training Hub image
                  "884aeeb039f9592252fe0c518acf31803a208757d4fa6500fe79884a022ea52d")

os.environ.setdefault("MLFLOW_EXPERIMENT_NAME", EXPERIMENT)
print("project:", NAMESPACE, "| volume:", WS, "| MLflow:", os.environ.get("MLFLOW_TRACKING_URI", "not enabled"))

## 1 · The memory estimator

In [ ]:
from training_hub.profiling.memory_estimator import estimate
MODEL = str(WS / "models/granite-4.1-3b")
for method in ["sft", "osft", "lora"]:
    try:
        print(method.upper(), estimate(training_method=method, model_path=MODEL, num_gpus=1,
                                       gpu_memory=24_000_000_000, max_tokens_per_gpu=4096))
    except Exception as e:
        print(method.upper(), "->", type(e).__name__, str(e)[:160])

In [ ]:
# What the estimator told us for Granite 4.1-3B on one GPU, against common GPU sizes
import matplotlib.pyplot as plt
need = {"Full SFT": (56, 73), "OSFT": (43, 55), "LoRA": (7.2, 9.1), "QLoRA (4-bit)": (3.0, 4.1)}
fig, ax = plt.subplots(figsize=(8, 3))
for i, (m, (lo, hi)) in enumerate(need.items()):
    ax.barh(i, hi, color="#c6c6c6"); ax.barh(i, lo, color="#0f62fe"); ax.text(hi + 1, i, f"{lo}–{hi} GiB", va="center")
for g, label in [(10, "share of an L4"), (24, "L4"), (48, "L40S"), (80, "H100")]:
    ax.axvline(g, color="#161616", lw=1, ls="--"); ax.text(g, len(need) - 0.4, label, rotation=90, va="bottom", fontsize=8)
ax.set_yticks(range(len(need)), need.keys()); ax.set_xlabel("GPU memory (GiB)"); ax.invert_yaxis(); plt.tight_layout()

**Our choice:** LoRA. It fits a ~10 GB share of one NVIDIA L4, trains in minutes, and the adapter is 62 MB.
Red Hat's LoRA path runs on **Unsloth**, the same library a dev team would use on its own.

## 2 · OSFT: full fine-tuning that keeps existing skills

Standard fine-tuning and LoRA can make a model forget what it knew. **OSFT** (built by Red Hat AI Innovation) finds
the parts of the weights that hold existing skills, freezes them, and trains the rest. Red Hat reports 48.4% vs 23.0%
for standard fine-tuning when a model learns eight tasks one after another.

Use it when the model must keep its general abilities while it learns your domain. We did not run it here: for this
model it needs 43–55 GiB, more than our GPU share. The call has the same shape as `sft`:

In [ ]:
RUN_OSFT = False      # needs a GPU with about 48 GB or more for Granite 4.1-3B

if RUN_OSFT:
    from training_hub import osft
    osft(model_path=MODEL,
         data_path=str(WS / "data/edgecopilot/train.jsonl"),
         ckpt_output_dir=str(WS / "checkpoints/edgecopilot-osft"),
         unfreeze_rank_ratio=0.3,          # train 30% of each layer's directions, protect the other 70%
         effective_batch_size=16, max_tokens_per_gpu=4096, max_seq_len=1024,
         num_epochs=2, learning_rate=5e-6)

## 3 · Pick the method by the mistake

| The model's mistake | Method | Training Hub | Our evidence |
|---|---|---|---|
| Wrong task or format | **SFT** (here LoRA) | `lora_sft` | Exact reports 6.3% → 92.7% |
| A gap in the training data | **More data, same recipe** | `lora_sft` | + 600 radio-style notes → 99.7–100% |
| The same mistake every time | **DPO** | TRL, in the same image | 92.7% → 99.7% |
| Breaks rules a checker can verify | **GRPO** | `lora_grpo` | Burst line 88% → 95% |
| Must keep general skills | **OSFT** | `osft` | Red Hat-reported, not run here |

GRPO learns only where the model is *sometimes* right: sample it a few times per problem first and check that the
scores differ. **Next:** notebook 03 trains.